In [9]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from qopt_funcs import *
from network_funcs import *

In [10]:
PARAMS = {
    'nu_tilde': 0.1,      # Source efficiency
    'p_det': 0.95,        # Detector efficiency
    'alpha': 0.18,        # Fiber loss (dB/km)
    'q_0': 0.01,          # Baseline QBER
    'nu_clock': 10**9     # Repetition rate (1 GHz)
}

In [11]:
N=10
beta = 2.6261                 # \beta param of S2 model
mu = 0.0233
A, dist = S2_graph_definite_N(N, beta, mu, D=2, sample_from_file=False, return_coords=False)

In [15]:
def build_prob_matrix(A, dist):
    T_matrix = 10**(-PARAMS['alpha'] * dist / 10) # Transmitivity matrix
    P_ent_matrix = PARAMS['nu_tilde'] * PARAMS['p_det'] * T_matrix # Probability of entanglement
    return P_ent_matrix * A # Multiply by Adjacency matrix to keep only existing edges

Probs_mtx = build_prob_matrix(A, dist)

In [34]:
def optimal_quantum_repeater_path(Probs_mtx, source, target=None,  P_BSM=0.5): # structured like nx.single_source_dijkstra
    W = np.zeros(np.shape(Probs_mtx))
    none_zero_edges = Probs_mtx>0
    W[none_zero_edges] = -np.log2(Probs_mtx[none_zero_edges]) - np.log2(P_BSM)
    G = nx.from_numpy_array(W)
    weights, paths = nx.single_source_dijkstra(G, source, target, weight='weight')
    #weights = np.exp(weights + np.log2(P_BSM))   # to avoid overcounting the Bell state success probability
    return weights, paths

weights, path = optimal_quantum_repeater_path(Probs_mtx, 0, target=5)
path

[0, 2, 5]

In [35]:
def calculate_expected_sequential_time(path, Probs_mtx, P_BSM=0.5):
    a = path[0] # First Node
    b = path[1] # Second Node
    T=1/Probs_mtx[a, b]
    for i in range(1,len(path)):
        a = path[i-1]
        b = path[i]
        T_i=1/Probs_mtx[a, b]
        T=(T+T_i)/P_BSM
    return T

calculate_expected_sequential_time(path, Probs_mtx)

np.float64(116.17214343921415)

## Post processing functions

In [8]:
def entanglement_generation_rate(total_time):
    return PARAMS['nu_clock']/total_time

def secret_key_rate(total_time, q, p_signal, p_darkcount):
    R_ent=entanglement_generation_rate(total_time)
    H = DV_keyrate(PARAMS['q_0'], p_signal, p_darkcount)
    return R_ent*(1-2*H)
